# Falnama — Analyst Review

A readable, compiled view of Falnama's research runs: what was screened, what moved,
whether wallets or news explain it, what the scenario cards say, and what the
recommender did with it.

### How to use it

1. **Put this notebook in the same folder as your Falnama outputs.** Any of these works:
   - the downloaded GitHub Actions artifact (`falnama-outputs-<number>.zip`) — **you don't need to unzip it**;
   - the unzipped artifact folder (the notebook can sit inside it or next to it);
   - a clone of the repo (anywhere in it, including `notebooks/`);
   - a folder of loose output files, with or without their sub-folders.
2. **Run all cells** (Jupyter: *Run → Run All Cells*; VS Code: *Run All*).

The notebook finds files by their names, so the folder layout doesn't matter. Several
downloads side by side are combined into one history. It needs only `pandas` and
`matplotlib` (both come with Anaconda). It doesn't need the `falnama` package or the
repo's config, and it never changes your files.

By default it shows the **latest** run. To look at another one, change `RUN` in the
next cell.

In [ ]:
# ---- Settings: the only cell you should need to edit ------------------------
RUN = "latest"        # or a date like "2026-11-05", or a run id like "20261105T153223Z"
DATA_FOLDER = None    # None = search the folder this notebook is in; or a path, e.g. "~/Downloads/falnama-outputs-123"
TOP_N = 15            # rows to show in the ranked tables
EXPORT_EXCEL = False  # True = also save this run's main tables to falnama_review_<run>.xlsx next to the notebook

## Locate the outputs
This cell holds the code that finds and reads the files. You don't need to read it,
but it is commented if you want to. It prints what it found and where.

In [ ]:
import io, json, os, re, zipfile
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from types import SimpleNamespace

import pandas as pd
from IPython.display import Markdown, display

try:
    import matplotlib.pyplot as plt
    from matplotlib.colors import LinearSegmentedColormap
except ImportError:      # tables still work without matplotlib; charts are skipped
    plt = None
    print("matplotlib is not installed: tables will show, charts will be skipped.")

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 90)

# Every Falnama output file carries the run id in its name: a compact UTC
# timestamp like 20261107T140618Z. The run manifest is the one file every run writes.
RUN_ID = re.compile(r"(\d{8}T\d{6}Z)")
MANIFEST = re.compile(r"^run_manifest_(\d{8}T\d{6}Z)\.json$")
SKIP_DIRS = {"__pycache__", "node_modules", "site-packages", "venv"}  # plus any hidden (.dot) folder
MAX_DEPTH = 6            # how many folder levels below the start folder to search


class Outputs:
    """Every run-stamped Falnama file found, indexed by FILE NAME.

    Run-stamped files are written once and never edited, so a file name means the
    same content wherever it turns up. That's why folder layout doesn't matter and
    why overlapping downloads can be combined safely. The '*_latest' copies are
    ignored on purpose: with several downloads you can't tell which one is "latest".
    """

    def __init__(self, start):
        self.start = Path(start)
        self.files = {}      # name -> ("disk", path) or ("zip", zip_path, member)
        self.places = []     # folders / zips that held run manifests
        self.unreadable = [] # files we found but couldn't read (e.g. cloud placeholders)
        self._zips = {}

    @property
    def runs(self):
        return sorted({m.group(1) for n in self.files if (m := MANIFEST.match(n))})

    def scan_folder(self, root):
        root = Path(root)
        zips = []
        for dirpath, dirnames, filenames in os.walk(root):
            depth = len(Path(dirpath).relative_to(root).parts)
            dirnames[:] = [] if depth >= MAX_DEPTH else sorted(
                d for d in dirnames if not d.startswith(".") and d not in SKIP_DIRS)
            for name in filenames:
                if name.lower().endswith(".zip"):
                    zips.append(Path(dirpath) / name)
                elif RUN_ID.search(name) and name.endswith((".csv", ".json")):
                    self.files.setdefault(name, ("disk", Path(dirpath) / name))
                    if MANIFEST.match(name):
                        self._note_place(Path(dirpath))
        for z in zips:            # after the loose files, so unzipped copies take priority
            self.scan_zip(z)

    def scan_zip(self, zpath):
        try:
            z = zipfile.ZipFile(zpath)
        except (zipfile.BadZipFile, OSError):
            return
        members = [n for n in z.namelist() if not n.endswith("/")]
        if not any(MANIFEST.match(Path(n).name) for n in members):
            return                # some other zip: leave it alone
        self._zips[zpath] = z
        for n in members:
            name = Path(n).name
            if RUN_ID.search(name) and name.endswith((".csv", ".json")):
                self.files.setdefault(name, ("zip", zpath, n))
        self.places.append(zpath)

    def _note_place(self, folder):
        place = folder.parent if folder.name == "run_logs" else folder
        if place not in self.places:
            self.places.append(place)

    def read_bytes(self, name):
        entry = self.files.get(name)
        if entry is None:
            return None
        try:
            if entry[0] == "disk":
                return entry[1].read_bytes()
            return self._zips[entry[1]].read(entry[2])
        except (OSError, KeyError, zipfile.BadZipFile) as exc:
            self.unreadable.append(f"{name}: {exc}")
            return None

    def where(self, place):
        try:
            return str(Path(place).relative_to(self.start)) or "."
        except ValueError:
            return str(place)


def notebook_folder():
    """Where this notebook lives. Jupyter starts in the notebook's folder; VS Code
    also tells us the notebook's path directly."""
    vsc = globals().get("__vsc_ipynb_file__")
    return Path(vsc).parent if vsc else Path.cwd()


def find_outputs(data_folder=None):
    start = Path(data_folder).expanduser().resolve() if data_folder else notebook_folder()
    if not start.is_dir():
        raise FileNotFoundError(f"DATA_FOLDER {str(start)!r} is not a folder.")
    out = Outputs(start)
    out.scan_folder(start)
    if not out.runs and data_folder is None:
        # Opened from inside the repo (e.g. notebooks/)? Look for a sibling outputs/ folder.
        for parent in list(start.parents)[:3]:
            if (parent / "outputs").is_dir():
                out.scan_folder(parent / "outputs")
            if out.runs:
                break
    if not out.runs:
        raise FileNotFoundError(
            f"No Falnama runs found in {start} (searched {MAX_DEPTH} folder levels down, "
            "including inside .zip files).\n"
            "  • Put this notebook in the folder that holds the outputs (or the downloaded "
            "falnama-outputs-*.zip), or\n"
            "  • set DATA_FOLDER in the settings cell to that folder's path.\n"
            "  A folder with outputs contains files named like run_manifest_20261107T140618Z.json."
        )
    return out


def run_day(run_id):
    return f"{run_id[:4]}-{run_id[4:6]}-{run_id[6:8]}"


def pick_run(out, wanted="latest"):
    """Accepts 'latest', a run id, a date ('2026-11-05' / '20261105'), or a pasted
    file name. A date with several runs picks that day's last one."""
    runs = out.runs
    if wanted in (None, "", "latest"):
        return runs[-1]
    text = str(wanted).strip()
    m = RUN_ID.search(text)
    if m and m.group(1) in runs:
        return m.group(1)
    digits = re.sub(r"\D", "", text)
    matches = [r for r in runs if len(digits) >= 8 and re.sub(r"\D", "", r).startswith(digits)]
    if matches:
        return matches[-1]
    raise ValueError(f"No run matches RUN={wanted!r}. Most recent runs: {', '.join(runs[-8:])}")


# ---- Readers: a missing or empty file comes back empty, never an error -------

def read_csv(name):
    raw = OUT.read_bytes(name)
    if not raw or not raw.strip():
        return pd.DataFrame()
    try:
        # Keep ids as text: some are numbers ("567688"), some are words ("mil-iran-strike").
        return pd.read_csv(io.BytesIO(raw), dtype={"market_id": str, "card_id": str})
    except pd.errors.EmptyDataError:
        return pd.DataFrame()


def read_json(name, default=None):
    raw = OUT.read_bytes(name)
    if not raw:
        return default
    try:
        return json.loads(raw)
    except ValueError:
        OUT.unreadable.append(f"{name}: not valid JSON")
        return default


def load_card(card_id):
    return read_json(f"{card_id}.json") if card_id else None


def load_run(run_id):
    """Everything one run wrote, as tables and dicts (see the field guide below)."""
    r = SimpleNamespace(id=run_id)
    j = lambda prefix, default: read_json(f"{prefix}_{run_id}.json", default)
    c = lambda prefix: read_csv(f"{prefix}_{run_id}.csv")
    r.manifest = j("run_manifest", {})
    r.health = j("pipeline_health", {})
    r.selection_diag = j("selection", {})          # Stage 1 counts + rejection reasons
    r.screen_diag = j("screen", {})                # Stage 1.5 LLM screen summary
    r.selected = c("relevant_markets")             # passed Stage 1 (keyword selector)
    r.rejected_markets = c("rejected_markets")     # failed Stage 1, with rejection_reason
    r.screen_verdicts = c("screen_verdicts")       # every screened market + LLM verdict
    r.screened = c("screened_markets")             # survivors of the screen
    r.fingerprints = c("market_fingerprints")      # Stage 2a wallet fingerprints per market
    r.wallet_evidence = c("wallet_evidence")       # every wallet examined, with facts
    r.ranked = c("ranked_anomalies")               # Stage 2 anomaly scores
    r.strong = c("strong_anomalies")
    r.concentration = c("concentration_diagnostics")
    r.news_lag = j("news_lag", []) or []           # Stage 5 public-information check
    r.recommendations = pd.DataFrame(j("recommendations", []) or [])
    r.rejected_signals = c("rejected_signals")
    r.orders = pd.DataFrame(j("simulated_orders", []) or [])
    # Scenario cards this run wrote or reused, plus any the recommender cited.
    cm = r.manifest.get("cards", {}) or {}
    ids = [Path(f).stem for f in cm.get("files", [])] + list(cm.get("reused_card_ids", []))
    for frame in (r.rejected_signals, r.recommendations):
        if "card_id" in frame:
            ids += [str(x) for x in frame["card_id"].dropna()]
    r.cards = {cid: card for cid in dict.fromkeys(ids) if (card := load_card(cid))}
    r.new_card_ids = [Path(f).stem for f in cm.get("files", [])]
    # One lookup table of market facts (name, url, close time, volume...) for joins.
    frames = [f for f in (r.screen_verdicts, r.selected) if not f.empty and "market_id" in f]
    r.markets = (pd.concat(frames).drop_duplicates("market_id").set_index("market_id")
                 if frames else pd.DataFrame())
    return r


OUT = find_outputs(DATA_FOLDER)
RUN_ID_PICKED = pick_run(OUT, RUN)
R = load_run(RUN_ID_PICKED)

runs = OUT.runs
print(f"Found {len(runs)} runs ({run_day(runs[0])} → {run_day(runs[-1])}) in:")
for p in OUT.places:
    print("   •", OUT.where(p))
print(f"\nShowing run {R.id}  ({R.manifest.get('run_time_utc', '?')})"
      + ("   ← the latest" if R.id == runs[-1] else f"   (the latest is {runs[-1]})"))

In [ ]:
# ---- Small display helpers and chart style ----------------------------------
# Colors: a fixed, colorblind-checked categorical order (blue, orange, aqua...) and a
# single-hue blue ramp for magnitudes. Charts are light-background static images.
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
MUTED = "#a3a29a"
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e1e0d9"
BLUES = None
if plt:
    BLUES = LinearSegmentedColormap.from_list("blues", ["#f4f8fd", "#86b6ef", "#2a78d6", "#0d366b"])
    plt.rcParams.update({
        "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
        "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK2, "axes.titlecolor": INK,
        "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
        "xtick.color": INK2, "ytick.color": INK2, "text.color": INK,
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
        "font.size": 9, "legend.frameon": False,
    })


def cols(df, wanted):
    """Only the columns that exist (older runs lack the newer fields)."""
    return [c for c in wanted if c in df.columns]


def show(df, wanted=None, n=None, sort=None, empty="— none this run —"):
    if df is None or len(df) == 0:
        display(Markdown(f"*{empty}*"))
        return
    view = df[cols(df, wanted)] if wanted else df
    if sort and sort in view:
        view = view.sort_values(sort, ascending=False)
    display(view.head(n) if n else view)


def truthy(series):
    return series.astype(str).str.strip().str.lower().isin({"true", "1", "yes"})


def no_urls(text):
    return re.sub(r"https?://\S+", "<url>", str(text))


def hours_between(earlier, later):
    try:
        a, b = pd.Timestamp(earlier), pd.Timestamp(later)
        return (b - a).total_seconds() / 3600
    except (ValueError, TypeError):
        return None


def label_bars(ax, bars, fmt="{:g}"):
    for b in bars:
        w = b.get_width()
        ax.text(w, b.get_y() + b.get_height() / 2, " " + fmt.format(w), va="center", color=INK2, fontsize=8)

## 1. Run at a glance
Each stage narrows the funnel. Zero recommendations is a **normal, valid** outcome:
the recommender deliberately leans towards not trading.

In [ ]:
m, h = R.manifest, R.health
g = lambda section, key, default=None: (m.get(section) or {}).get(key, default)

modes = {
    "market data": m.get("data_source"),
    "LLM screen": (g("screen", "mode") if g("screen", "enabled") else "off") if "screen" in m else "not in this run",
    "wallet fingerprints": ("on" if g("wallet_fingerprints", "enabled") else "off") if "wallet_fingerprints" in m else "not in this run",
    "scenario cards": m.get("card_mode"),
    "news-lag": (g("newslag", "mode") if g("newslag", "enabled") else "off") if "newslag" in m else "not in this run",
    "execution": g("execution", "mode", "—"),
}
display(Markdown(
    f"**Run {R.id}** · pipeline success: **{h.get('success', '?')}** · "
    + " · ".join(f"{k}: `{v}`" for k, v in modes.items())))
for kind in ("errors", "warnings"):
    for item in h.get(kind) or []:
        print(f"{kind[:-1].upper()}: {item}")

funnel = [
    ("markets pulled", g("market_selector", "input_markets")),
    ("passed keyword selector", g("market_selector", "selected_count")),
    ("kept by LLM screen", g("screen", "kept")),
    ("anomaly-scored", g("anomaly_detector", "markets_scored")),
    ("strong anomalies", g("anomaly_detector", "strong_count")),
    ("paper recommendations", g("recommender", "recommended_count")),
    ("simulated orders", g("execution", "simulated_orders")),
]
funnel = [(k, v) for k, v in funnel if v is not None]
side = {
    "scenario cards written (new)": g("cards", "cards_written"),
    "scenario cards reused": g("cards", "cards_reused"),
    "rejected signals": g("recommender", "rejected_count"),
    "news-lag assessed": g("newslag", "assessed"),
    "…still unexplained by prior news": g("newslag", "unexplained_residual"),
}
display(pd.DataFrame({"count": dict(funnel) | {k: v for k, v in side.items() if v is not None}}))

if plt and funnel:
    fig, ax = plt.subplots(figsize=(8, 0.42 * len(funnel) + 0.6))
    labels, values = zip(*funnel)
    bars = ax.barh(labels, values, color=SERIES[0], height=0.6)
    label_bars(ax, bars)
    ax.invert_yaxis(); ax.grid(axis="y", visible=False)
    ax.set_xlabel("markets")
    ax.set_title("Funnel for this run")
    plt.tight_layout(); plt.show()

## 2. Sensor health: is every input actually working?
A quiet run might be a genuinely quiet market, or a data source that silently went
dark. This table tells the two apart. **DARK** means the sensor returned nothing
usable, so read "no red flags" there as "unknown", not "clean". The pipeline's own
health report doesn't catch all of these.

In [ ]:
def history_table():
    """One row per run: the cross-run numbers used here and in section 9."""
    rows = []
    for rid in OUT.runs:
        mm = read_json(f"run_manifest_{rid}.json", {}) or {}
        hh = read_json(f"pipeline_health_{rid}.json", {}) or {}
        s = lambda sec, key: (mm.get(sec) or {}).get(key)
        tiers = s("wallet_fingerprints", "tiers") or {}
        examined = sum(tiers.values()) if tiers else None
        rows.append({
            "run_id": rid, "day": run_day(rid),
            "data_source": mm.get("data_source"), "card_mode": mm.get("card_mode"),
            "success": hh.get("success"), "errors": len(hh.get("errors") or []),
            "warnings": len(hh.get("warnings") or []),
            "pulled": s("market_selector", "input_markets"),
            "selected": s("market_selector", "selected_count"),
            "screen_kept": s("screen", "kept"),
            "screen_error": bool(s("screen", "error")),
            "price_failures": s("price_history", "markets_failed"),
            "scored": s("anomaly_detector", "markets_scored"),
            "strong": s("anomaly_detector", "strong_count"),
            "concentration_flags": s("anomaly_detector", "concentration_red_flags"),
            "fingerprint_flags": s("anomaly_detector", "fingerprint_red_flags"),
            "wallets_examined": s("wallet_fingerprints", "wallets_examined"),
            "wallet_data_available_pct": (round(100 * (1 - tiers.get("unavailable", 0) / examined), 1)
                                          if examined else None),
            "cards_written": s("cards", "cards_written"),
            "cards_reused": s("cards", "cards_reused"),
            "recommended": s("recommender", "recommended_count"),
            "rejected_signals": s("recommender", "rejected_count"),
            "simulated_orders": s("execution", "simulated_orders"),
            "newslag_assessed": s("newslag", "assessed"),
            "newslag_unexplained": s("newslag", "unexplained_residual"),
            "newslag_errors": s("newslag", "checker_errors"),
        })
    return pd.DataFrame(rows)


HIST = history_table()


def dark_since(column, run_id, threshold=50):
    """First run of the current unbroken streak where `column` < threshold, if any."""
    col = HIST.set_index("run_id")[column].loc[:run_id].dropna()
    if col.empty or col.iloc[-1] >= threshold:
        return None
    streak = col[::-1].lt(threshold).cumprod().astype(bool)
    first = streak[streak].index[-1]
    return f"since {run_day(first)} ({int(streak.sum())} run(s) in a row)"


def sensor_health(r):
    rows = []
    add = lambda sensor, status, detail: rows.append({"sensor": sensor, "status": status, "detail": detail})
    m, h = r.manifest, r.health
    s = lambda sec: m.get(sec) or {}

    add("Pipeline", "✅ OK" if h.get("success") else "⛔ FAILED",
        f"{len(h.get('errors') or [])} error(s), {len(h.get('warnings') or [])} warning(s)")
    src = m.get("data_source")
    add("Market data", "✅ OK" if src == "live" else "⚠️ CHECK",
        f"{s('market_selector').get('input_markets', '?')} markets pulled from "
        + ("live Polymarket" if src == "live" else f"'{src}' (demo/fixture data, not live)"))

    sc = s("screen")
    if not sc or not sc.get("enabled"):
        add("LLM relevance screen", "— off", "screen disabled; every Stage-1 market went on")
    elif sc.get("error"):
        add("LLM relevance screen", "⛔ FAILED OPEN", f"kept everything unscreened: {sc['error']}")
    else:
        add("LLM relevance screen", "✅ OK" if sc.get("mode") == "live" else "⚠️ MOCK",
            f"{sc.get('kept')} kept / {sc.get('dropped')} dropped"
            + ("" if sc.get("mode") == "live" else " — mock verdicts are placeholders"))

    ph = s("price_history")
    if ph:
        fails = ph.get("markets_failed", 0)
        add("Price history (CLOB)", "✅ OK" if not fails else "⚠️ CHECK",
            f"{ph.get('markets_requested')} requested, {fails} failed and skipped")

    for name, frame, avail_col, reason_col in [
        ("Wallet concentration", r.concentration, "concentration_available", "concentration_reason"),
        ("Wallet fingerprints", r.fingerprints, "fingerprint_available", "fingerprint_reason"),
    ]:
        if frame.empty or avail_col not in frame:
            if name == "Wallet fingerprints" and not s("wallet_fingerprints").get("enabled", False):
                add(name, "— off", "not enabled in this run")
            else:
                add(name, "⚠️ CHECK", "no diagnostics file for this run")
            continue
        ok = truthy(frame[avail_col])
        pct = 100 * ok.mean()
        top = Counter(no_urls(x) for x in frame.loc[~ok, reason_col].dropna()).most_common(1) if reason_col in frame else []
        why = f" — most common reason: {top[0][0]!r} ({top[0][1]}×)" if top else ""
        streak = dark_since("wallet_data_available_pct", r.id) if name == "Wallet fingerprints" else None
        status = "✅ OK" if pct >= 90 else ("⚠️ DEGRADED" if pct >= 50 else "⛔ DARK")
        add(name, status, f"data for {ok.sum()}/{len(frame)} markets ({pct:.0f}%){why}"
            + (f" — dark {streak}" if streak else ""))

    cm = s("cards")
    if cm:
        mock_in_live = [cid for cid, c in r.cards.items() if c.get("mock_mode") and m.get("card_mode") == "live"]
        bad_hash = cm.get("hash_mismatch_card_ids") or []
        status = "⛔ CHECK" if bad_hash else ("⚠️ CHECK" if mock_in_live else ("✅ OK" if m.get("card_mode") == "live" else "⚠️ MOCK"))
        detail = f"{cm.get('cards_written', 0)} new + {cm.get('cards_reused', 0)} reused ({m.get('card_mode')})"
        if bad_hash:
            detail += f"; {len(bad_hash)} card(s) FAILED their integrity hash: {bad_hash}"
        if mock_in_live:
            detail += f"; {len(mock_in_live)} mock placeholder card(s) used in a live run: {mock_in_live}"
        add("Scenario cards", status, detail)

    nl = s("newslag")
    if nl:
        if not nl.get("enabled"):
            add("News-lag check", "— off", "strong anomalies are not checked against prior news")
        else:
            errs = nl.get("checker_errors", 0)
            strong = s("anomaly_detector").get("strong_count", 0)
            add("News-lag check", "⚠️ CHECK" if errs or nl.get("mode") != "live" else "✅ OK",
                f"{nl.get('assessed')} of {strong} strong anomalies assessed ({nl.get('mode')}), "
                f"{errs} checker error(s) — an error leaves the move UNEXPLAINED, never dismissed")
    ex = s("execution")
    if ex:
        add("Execution (paper only)", "✅ OK", f"mode {ex.get('mode')}, {ex.get('simulated_orders', 0)} simulated order(s)")
    return pd.DataFrame(rows)


with pd.option_context("display.max_colwidth", 200):
    display(sensor_health(R).style.hide(axis="index").set_properties(**{"text-align": "left"}))
if OUT.unreadable:
    print("Files found but not readable (cloud-sync placeholders? download them and re-run):")
    for line in OUT.unreadable[:10]:
        print("   ", line)

## 3. Market selection (Stage 1) and the LLM relevance screen (Stage 1.5)

**Stage 1** is a cheap keyword selector: a `relevance_score` by topic, plus hard rejects,
a minimum volume, a filter that drops already-resolved markets, and diversity caps.
**Stage 1.5** asks an LLM to rate each surviving market on two **independent** 0–100 axes.
A market has to clear **both** floors:

| field | question it answers |
|---|---|
| `screen_economic_salience` | Would this outcome move public assets (stocks, FX, commodities)? |
| `screen_information_asymmetry` | Could a small group plausibly know first? |
| `screen_verdict` / `screen_drop_reason` | keep/drop, and which floor it missed |
| `screen_rationale` | the model's one-line reasoning (the main thing to check by eye) |

In [ ]:
sd, scd = R.selection_diag or {}, R.screen_diag or {}
if sd.get("rejection_reasons"):
    # Collapse run-specific details (dates, scores, slugs) so similar reasons group together.
    def reason_kind(text):
        text = re.sub(r"\(close_time [^)]*\)", "(close_time in the past)", text)
        text = re.sub(r"relevance score [\d.]+ < [\d.]+", "relevance score below floor", text)
        text = re.sub(r"volume [\d.]+ < [\d.]+", "volume below floor", text)
        return re.sub(r"event_family=\S+", "event_family=<same event>", text)
    kinds = Counter()
    for reason, n in sd["rejection_reasons"].items():
        kinds[reason_kind(reason)] += n
    print("Why markets were rejected at Stage 1:")
    display(pd.Series(kinds, name="markets").sort_values(ascending=False).to_frame())

v = R.screen_verdicts
if not v.empty and "screen_verdict" in v:
    keep = v["screen_verdict"].eq("keep")
    print(f"LLM screen: {keep.sum()} kept, {(~keep).sum()} dropped "
          f"(floors: salience ≥ {scd.get('min_economic_salience', '?')}, asymmetry ≥ {scd.get('min_information_asymmetry', '?')})")
    if plt:
        fig, ax = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios": [1.1, 1]})
        x, y = v["screen_economic_salience"], v["screen_information_asymmetry"]
        ax[0].scatter(x[~keep], y[~keep], s=22, color=MUTED, alpha=0.7, label="dropped", linewidths=0)
        ax[0].scatter(x[keep], y[keep], s=26, color=SERIES[0], label="kept", edgecolors="#fcfcfb", linewidths=0.8)
        for val, axis_line in ((scd.get("min_economic_salience"), ax[0].axvline), (scd.get("min_information_asymmetry"), ax[0].axhline)):
            if val is not None:
                axis_line(val, color=INK2, linestyle="--", linewidth=1)
        ax[0].set_xlabel("economic salience"); ax[0].set_ylabel("information asymmetry")
        ax[0].set_xlim(-3, 103); ax[0].set_ylim(-3, 103)
        ax[0].set_title("Screen verdicts (dashed lines = floors)"); ax[0].legend(loc="lower right")
        topics = (v.assign(kept=keep).groupby("screen_topic")["kept"].agg(["sum", "count"])
                    .sort_values("count"))
        bars = ax[1].barh(topics.index, topics["count"] - topics["sum"], color=MUTED, height=0.6, label="dropped")
        ax[1].barh(topics.index, topics["sum"], left=topics["count"] - topics["sum"], color=SERIES[0],
                   height=0.6, label="kept", edgecolor="#fcfcfb", linewidth=1.5)
        ax[1].grid(axis="y", visible=False); ax[1].set_title("Screened markets by LLM topic")
        ax[1].set_xlabel("markets"); ax[1].legend(loc="lower right")
        plt.tight_layout(); plt.show()

    print("Markets the screen KEPT (highest asymmetry first):")
    show(v[keep], ["market_name", "screen_topic", "screen_economic_salience", "screen_information_asymmetry",
                   "screen_rationale"], n=TOP_N, sort="screen_information_asymmetry")
    print("Closest calls among DROPPED markets (worth a sanity check):")
    dropped = v[~keep].assign(margin=lambda d: d[["screen_economic_salience", "screen_information_asymmetry"]].min(axis=1))
    show(dropped, ["market_name", "screen_drop_reason", "screen_economic_salience", "screen_information_asymmetry",
                   "screen_rationale"], n=10, sort="margin")
elif not R.selected.empty:
    print("This run has no LLM screen file. Stage-1 selections:")
    show(R.selected, ["market_name", "primary_topic", "country_or_region", "relevance_score"], n=TOP_N, sort="relevance_score")

## 4. Anomalies (Stage 2)

Each scored market gets a 0–100 composite from four sub-scores, plus small bonuses:

| field | meaning |
|---|---|
| `score_magnitude` (35%) | how big the price move was (a 10-point move scores high) |
| `score_speed` (25%) | how fast it happened |
| `score_persistence` (20%) | whether the move held afterwards (blank = too recent to tell; the composite reweights) |
| `score_unusualness` (20%) | how rare a move of that size is for this market |
| `time_to_close_bonus` | extra weight when the market resolves soon |
| `concentration_bonus` / `fingerprint_bonus` | wallet red flags (section 5) |
| `price_before` → `price_after` | probability before the trigger and at the peak of the move |
| `anomaly_class` | strong ≥ 85, medium ≥ 70, weak ≥ 50; only **strong** goes on to cards and the recommender |

In [ ]:
a = R.ranked
if a.empty:
    display(Markdown("*No anomalies were scored this run.*"))
else:
    a = a.assign(move_pts=lambda d: ((d["price_after"] - d["price_before"]) * 100).round(1))
    show(a, ["market_name", "anomaly_class", "anomaly_score", "price_before", "price_after", "move_pts",
             "anomaly_trigger_time_utc", "fingerprint_tier", "concentration_tier"], n=TOP_N, sort="anomaly_score")
    if plt:
        top = a.sort_values("anomaly_score", ascending=False).head(TOP_N)
        parts = cols(top, ["score_magnitude", "score_speed", "score_persistence", "score_unusualness",
                           "time_to_close_bonus", "concentration_bonus", "fingerprint_bonus", "anomaly_score"])
        grid = top.set_index("market_name")[parts].astype(float)
        fig, ax = plt.subplots(1, 2, figsize=(13, 0.34 * len(grid) + 1.4), gridspec_kw={"width_ratios": [3, 1]})
        ax[0].imshow(grid.values, cmap=BLUES, vmin=0, vmax=100, aspect="auto")
        ax[0].set_xticks(range(len(parts)), [p.replace("score_", "").replace("_", " ") for p in parts], rotation=30, ha="right")
        ax[0].set_yticks(range(len(grid)), [n[:55] for n in grid.index]); ax[0].grid(False)
        for i in range(grid.shape[0]):
            for j in range(grid.shape[1]):
                val = grid.iat[i, j]
                if pd.notna(val):
                    ax[0].text(j, i, f"{val:.0f}", ha="center", va="center", fontsize=7.5,
                               color="#fcfcfb" if val > 55 else INK)
        ax[0].set_title(f"What drives the top {len(grid)} scores (blank = not measured)")
        order = ["strong", "medium", "weak", "subthreshold"]
        counts = a["anomaly_class"].value_counts().reindex(order).fillna(0)
        bars = ax[1].barh(counts.index, counts.values, color=SERIES[0], height=0.6)
        label_bars(ax[1], bars); ax[1].invert_yaxis(); ax[1].grid(axis="y", visible=False)
        ax[1].set_title("Markets by class")
        plt.tight_layout(); plt.show()

## 5. Wallet evidence: concentration and insider-style fingerprints (Stage 2a)

**Concentration**: is a liquid market being moved by only a few wallets? Measured by
top-1 and top-3 wallet share and the Gini coefficient of volume across wallets.

**Fingerprints**: a wallet **matches** if it is all four of: *long-shot* (bought at
≤ 35¢), *large* (≥ $10K staked), *fresh* (first-ever Polymarket activity ≤ 14 days
before the bet), and *focused* (≤ 25 markets traded in its lifetime). Market tiers:

| `fingerprint_tier` | meaning |
|---|---|
| `cluster` | ≥ 3 matching wallets on the **same side**: a red flag that adds to the anomaly score |
| `watch` | 1–2 matches: worth reviewing, no score effect (lone matches also turn up in innocent markets) |
| `none` | wallets examined, nothing matched |
| `unavailable` | the data couldn't be fetched, so this is **unknown**, not "clean" |

In [ ]:
fp, we = R.fingerprints, R.wallet_evidence
if fp.empty:
    display(Markdown("*No wallet-fingerprint file for this run.*"))
else:
    print("Fingerprint tiers this run:", dict(fp["fingerprint_tier"].value_counts()))
    flagged = fp[fp["fingerprint_tier"].isin(["cluster", "watch"])]
    print("\nMarkets with matching wallets:")
    show(flagged, ["market_name", "fingerprint_tier", "matched_wallets", "matched_stake_usd", "matched_outcome",
                   "candidate_wallets", "window_trade_usd", "fingerprint_reason"], sort="matched_wallets",
         empty="— no cluster or watch markets this run —")
    if not we.empty:
        print("Wallet evidence (matching wallets first). Open profile_url to see a wallet on Polymarket:")
        we2 = we.assign(_m=truthy(we["fingerprint_match"])).sort_values(["_m", "stake_usd"], ascending=False)
        show(we2, ["market_name", "fingerprint_match", "wallet_name", "outcome", "stake_usd", "avg_entry_price",
                   "payoff_multiple", "wallet_age_days", "markets_traded", "is_longshot", "is_large", "is_fresh",
                   "is_focused", "profile_url"], n=TOP_N)

conc = R.concentration
if not conc.empty and truthy(conc.get("concentration_available", pd.Series(dtype=str))).any():
    print("Most concentrated markets (top-3 wallet share):")
    show(conc[truthy(conc["concentration_available"])],
         ["market_name", "concentration_tier", "concentration_red_flag", "top1_wallet_share", "top3_wallet_share",
          "wallet_gini"], n=10, sort="top3_wallet_share")
elif not conc.empty:
    print("No wallet-concentration data this run (see Sensor health above).")

## 6. Was it already public? (news-lag, Stage 5)

For each strong anomaly, Falnama retrieves news published **before** the trigger time
and asks an LLM how strong that public information already was. Only earlier news can
*explain* a move. Coverage is placed on a ladder, from weakest to strongest:
`none` → `market_commentary` → `retrospective` → `speculation` → `rumor` → `denial` →
`official_announcement` → `confirmation`.

- `public_information_score` (0–100): how much of the move public news already explains.
- `residual_anomaly_score` = anomaly score × (1 − public/100): **the "genuinely surprising"
  ranking**. High residual means a big move that prior news doesn't explain.

In [ ]:
nl = pd.DataFrame(R.news_lag)
if nl.empty:
    display(Markdown("*No news-lag assessments this run (stage off, or no strong anomalies).*"))
else:
    nl["market_id"] = nl["market_id"].astype(str)
    if not R.ranked.empty:
        nl = nl.merge(R.ranked[["market_id", "anomaly_score"]], on="market_id", how="left")
    nl["articles_found"] = nl["evidence"].apply(lambda e: len(e or []))
    nl["earliest_article_h_before"] = [
        max((hours_between(x.get("published_time_utc"), t) or 0 for x in (e or [])), default=None)
        for e, t in zip(nl["evidence"], nl["anomaly_trigger_time_utc"])]
    nl["earliest_article_h_before"] = nl["earliest_article_h_before"].round(1)
    show(nl, ["market_name", "anomaly_score", "public_information_score", "residual_anomaly_score",
              "information_state", "articles_found", "earliest_article_h_before", "anomaly_trigger_time_utc"],
         sort="residual_anomaly_score")
    if plt and "anomaly_score" in nl:
        d = nl.sort_values("residual_anomaly_score")
        fig, ax = plt.subplots(figsize=(9, 0.5 * len(d) + 1))
        ys = range(len(d))
        ax.barh([y + 0.2 for y in ys], d["anomaly_score"], height=0.38, color=MUTED, label="anomaly score")
        bars = ax.barh([y - 0.2 for y in ys], d["residual_anomaly_score"], height=0.38, color=SERIES[0],
                       label="residual after prior news")
        label_bars(ax, bars, "{:.0f}")
        ax.set_yticks(list(ys), [n[:50] for n in d["market_name"]]); ax.grid(axis="y", visible=False)
        ax.set_xlim(0, 105); ax.set_title("How much of each strong move is left unexplained by prior news")
        ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2); plt.tight_layout(); plt.show()

## 7. Scenario cards and what the recommender did with them

A **scenario card** is a frozen, content-hashed hypothesis written *before* acting:
which tradable assets the market maps to, the expected direction and size over 12 hours,
and a confidence. A card can be **reused** for up to 7 days if the market hasn't changed.
The recommender only acts on a **strong** anomaly (and only the top-ranked share of those)
with an eligible card created **before** the trigger. Anything else lands in *rejected signals*, with the reason.

In [ ]:
def card_predictions(cards):
    rows = []
    for cid, c in cards.items():
        for p in c.get("predictions") or [{}]:
            elig = p.get("trade_eligibility") or {}
            ci = p.get("confidence_interval_bps") or [None, None]
            rows.append({
                "card_id": cid, "market_id": str((c.get("source") or {}).get("market_id", "")),
                "market_name": c.get("market_name"), "created_utc": c.get("created_time_utc"),
                "mock": c.get("mock_mode"), "new_this_run": cid in R.new_card_ids,
                "ticker": p.get("ticker"), "asset": p.get("asset"), "asset_class": p.get("asset_class"),
                "direction": p.get("expected_direction"), "expected_bps_12h": p.get("expected_return_12h_bps"),
                "ci_low_bps": ci[0], "ci_high_bps": ci[-1], "confidence": p.get("confidence"),
                "eligible": elig.get("eligible"), "eligibility_reason": elig.get("reason"),
            })
    return pd.DataFrame(rows)


CARDS = card_predictions(R.cards)
cards_listed = sum((R.manifest.get("cards") or {}).get(k, 0) or 0 for k in ("cards_written", "cards_reused"))
if CARDS.empty and cards_listed:
    display(Markdown(f"*This run used {cards_listed} scenario card(s), but their files (normally in `index_cards/`) "
                     "aren't in this folder, so they can't be shown.*"))
elif CARDS.empty:
    display(Markdown("*No scenario cards for this run.*"))
else:
    print(f"{len(R.cards)} card(s) in play this run ({len(R.new_card_ids)} newly written); "
          f"{int(CARDS['eligible'].astype(str).eq('True').sum())} of {len(CARDS)} asset predictions are trade-eligible.")
    show(CARDS.sort_values(["new_this_run", "market_name"], ascending=[False, True]),
         ["market_name", "new_this_run", "created_utc", "mock", "ticker", "direction", "expected_bps_12h",
          "ci_low_bps", "ci_high_bps", "confidence", "eligible", "eligibility_reason"])

print("Paper recommendations:")
show(R.recommendations, empty="— none this run (the recommender found nothing that cleared every check) —")
print("Rejected signals (why each candidate was not recommended):")
show(R.rejected_signals, ["market_name", "anomaly_score", "anomaly_trigger_time_utc", "card_id", "reason"])
if not R.orders.empty:
    print("Simulated (paper) orders:")
    show(R.orders)

## 8. Dossier: one page per strong anomaly
Everything Falnama knows about each strong move, from every stage, in one place. This
is usually the best starting point for a human review.

In [ ]:
def strong_history():
    """Every market ever flagged strong, across all runs found: first/last day, how often."""
    frames = []
    for rid in OUT.runs:
        s = read_csv(f"strong_anomalies_{rid}.csv")
        if not s.empty and "market_id" in s:
            frames.append(s[cols(s, ["market_id", "market_name", "anomaly_score"])].assign(run_id=rid))
    if not frames:
        return pd.DataFrame()
    s = pd.concat(frames)
    return (s.groupby("market_id").agg(market_name=("market_name", "last"), runs_flagged=("run_id", "nunique"),
                                        first_flagged=("run_id", "min"), last_flagged=("run_id", "max"),
                                        max_score=("anomaly_score", "max"))
              .sort_values("last_flagged", ascending=False)
              .assign(first_day=lambda d: d["first_flagged"].map(run_day), last_day=lambda d: d["last_flagged"].map(run_day)))


STRONG_HIST = strong_history()


def fmt(x, nd=1):
    return "—" if x is None or (isinstance(x, float) and pd.isna(x)) else (f"{x:.{nd}f}" if isinstance(x, float) else str(x))


def dossier(market_id, r=None):
    r = r or R
    market_id = str(market_id)
    out = []
    row = r.ranked[r.ranked["market_id"] == market_id] if not r.ranked.empty else pd.DataFrame()
    info = r.markets.loc[market_id] if market_id in r.markets.index else pd.Series(dtype=object)
    name = (row["market_name"].iloc[0] if not row.empty else info.get("market_name", market_id))
    if row.empty:
        out.append(f"### {name}\n*Not scored this run.*")
    else:
        a = row.iloc[0]
        out.append(f"### {name}\n**anomaly score {fmt(a.anomaly_score)} ({a.anomaly_class})** · market `{market_id}`"
                   + (f" · [open on Polymarket]({info.get('market_url')})" if isinstance(info.get("market_url"), str) else ""))
        move = (a.price_after - a.price_before) * 100
        out.append(f"- **Move:** {a.price_before:.2f} → {a.price_after:.2f} ({move:+.1f} pts) triggered "
                   f"{a.anomaly_trigger_time_utc}; market closes {info.get('close_time', '?')}, volume "
                   f"${float(info.get('volume', 0) or 0):,.0f}")
        out.append("- **Sub-scores:** " + " · ".join(
            f"{k.replace('score_', '')} {fmt(a.get(k), 0)}" for k in
            ["score_magnitude", "score_speed", "score_persistence", "score_unusualness"] if k in a)
            + " · bonuses: " + ", ".join(f"{k.replace('_bonus', '').replace('_', ' ')} +{fmt(a.get(k), 0)}"
                                          for k in ["time_to_close_bonus", "concentration_bonus", "fingerprint_bonus"] if k in a))
        out.append(f"- **Wallets:** fingerprint `{a.get('fingerprint_tier', '—')}` "
                   f"({fmt(a.get('fingerprint_matched_wallets'), 0)} matched) · concentration "
                   f"`{a.get('concentration_tier', '—')}`, {no_urls(a.get('concentration_reason', ''))[:120]}")
    if "screen_rationale" in info and isinstance(info.get("screen_rationale"), str):
        out.append(f"- **Screen:** salience {fmt(info.get('screen_economic_salience'), 0)}, asymmetry "
                   f"{fmt(info.get('screen_information_asymmetry'), 0)} ({info.get('screen_topic')}). "
                   f"*{info.get('screen_rationale')}*")
    for n in r.news_lag:
        if str(n.get("market_id")) == market_id:
            out.append(f"- **Prior news:** strongest state `{n.get('information_state')}`; public-information "
                       f"{fmt(n.get('public_information_score'), 0)} → **residual {fmt(n.get('residual_anomaly_score'))}**. "
                       f"*{n.get('uncertainty_note', '')}*")
            for e in (n.get("evidence") or [])[:4]:
                hb = hours_between(e.get("published_time_utc"), n.get("anomaly_trigger_time_utc"))
                out.append(f"    - {fmt(hb, 0)}h before · `{e.get('information_state')}` · "
                           f"[{(e.get('snippet') or '').strip()[:90]}]({e.get('url')}) ({e.get('source')})")
    cards = CARDS[CARDS["market_id"] == market_id] if not CARDS.empty else pd.DataFrame()
    for cid, grp in (cards.groupby("card_id") if not cards.empty else []):
        c = grp.iloc[0]
        when = ""
        if not row.empty:
            lead = hours_between(c.created_utc, row.iloc[0].anomaly_trigger_time_utc)
            when = f", {abs(lead):.0f}h {'BEFORE' if lead >= 0 else 'AFTER'} the trigger" if lead is not None else ""
        out.append(f"- **Card** `{cid}` ({'mock' if str(c.mock) == 'True' else 'live'}, created {c.created_utc}{when}):")
        for _, p in grp.iterrows():
            out.append(f"    - {p.ticker} ({p.asset_class}): {p.direction}, {fmt(p.expected_bps_12h, 0)} bps "
                       f"[{fmt(p.ci_low_bps, 0)}, {fmt(p.ci_high_bps, 0)}], confidence {fmt(p.confidence, 2)}"
                       f" — {'ELIGIBLE' if str(p.eligible) == 'True' else 'not eligible: ' + str(p.eligibility_reason)}")
    if cards.empty and not row.empty:
        out.append("- **Card:** none in play this run")
    rec, rej = r.recommendations, r.rejected_signals
    outcomes = []
    if not rec.empty and "market_id" in rec and (rec["market_id"].astype(str) == market_id).any():
        outcomes.append("✅ paper recommendation issued (see section 7)")
    if not rej.empty and "market_id" in rej:
        outcomes += [f"not recommended — {why}" for why in rej.loc[rej["market_id"].astype(str) == market_id, "reason"]]
    if not outcomes and not row.empty and row.iloc[0].anomaly_class == "strong":
        outcomes.append("not evaluated by the recommender — it only considers the top-ranked share of strong "
                        "anomalies (`recommend.strong_rank_percentile_cutoff` in the config; 0.50 = top half)")
    out += [f"- **Outcome:** {o}" for o in outcomes]
    if not STRONG_HIST.empty and market_id in STRONG_HIST.index:
        sh = STRONG_HIST.loc[market_id]
        out.append(f"- **History:** flagged strong in {sh.runs_flagged} run(s), {run_day(sh.first_flagged)} → "
                   f"{run_day(sh.last_flagged)} (max score {fmt(sh.max_score)})")
    display(Markdown("\n".join(out)))


if R.strong.empty:
    display(Markdown("*No strong anomalies this run. Use `dossier(\"<market_id>\")` to look at any scored market.*"))
else:
    for mid in R.strong.sort_values("anomaly_score", ascending=False)["market_id"]:
        dossier(mid)

## 9. Trends across runs
How the numbers move over time, from every run found. Useful for spotting drift, the
effect of a config change, or a sensor that went quiet.

In [ ]:
show(HIST.drop(columns=["run_id"]).tail(TOP_N).iloc[::-1])
if plt and len(HIST) > 1:
    panels = [("selected", "passed keyword selector"), ("screen_kept", "kept by LLM screen"),
              ("strong", "strong anomalies"), ("wallet_data_available_pct", "wallet data available (%)"),
              ("cards_written", "new scenario cards"), ("recommended", "paper recommendations")]
    panels = [p for p in panels if HIST[p[0]].notna().any()]
    days = pd.to_datetime(HIST["day"])
    fig, axes = plt.subplots(2, 3, figsize=(13, 5.5), sharex=True)
    for ax, (col, title) in zip(axes.flat, panels):
        ax.plot(days, HIST[col], color=SERIES[0], linewidth=2, marker="o", markersize=3)
        here = HIST["run_id"] == R.id
        ax.plot(days[here], HIST.loc[here, col], "o", color=SERIES[1], markersize=8, label="this run")
        ax.set_title(title); ax.set_ylim(bottom=0); ax.tick_params(axis="x", rotation=30)
    for ax in list(axes.flat)[len(panels):]:
        ax.set_visible(False)
    fig.suptitle("Each run (blue); the run shown in this notebook is the orange dot", x=0.01, ha="left", color=INK2, fontsize=9)
    plt.tight_layout(); plt.show()

print("Markets flagged strong most recently (across all runs found):")
show(STRONG_HIST.reset_index(), ["market_name", "runs_flagged", "first_day", "last_day", "max_score"], n=TOP_N)

## 10. Detection backtest (only if backtest files are present)
`python scripts/backtest.py` writes these into `outputs/backtest/`. They aren't part of
the daily artifact, so this section is often empty.

In [ ]:
bt_runs = sorted({m.group(1) for n in OUT.files if (m := re.match(r"^detection_(\d{8}T\d{6}Z)\.csv$", n))})
if not bt_runs:
    display(Markdown("*No backtest files found.*"))
else:
    b = bt_runs[-1]
    print(f"Latest backtest: {b}")
    summary = read_json(f"summary_{b}.json", {}) or {}
    if summary:
        display(pd.json_normalize(summary).T.rename(columns={0: "value"}))
    show(read_csv(f"detection_{b}.csv"), ["label", "market_name", "resolved_outcome", "peak_anomaly_score",
         "detected_strong", "lead_hours", "fingerprint_tier", "fingerprint_matched_wallets", "error"])

## 11. Look things up yourself
Everything is already loaded. A few handy one-liners (paste them into a new cell):

```python
dossier("567688")                       # the one-page view for any market id in this run
find("iran")                            # search this run's markets by name
print(json.dumps(R.cards["<card_id>"], indent=2))   # read a whole scenario card
R2 = load_run(pick_run(OUT, "2026-11-01"))           # load another run alongside this one
R.screen_verdicts, R.ranked, R.wallet_evidence, R.news_lag, R.manifest   # raw tables
```

In [ ]:
def find(text, r=None):
    """Markets in this run whose name contains `text` (any case), with their status at each stage."""
    r = r or R
    if r.markets.empty:
        return pd.DataFrame()
    hit = r.markets[r.markets["market_name"].str.contains(text, case=False, na=False)].copy()
    if not r.ranked.empty:
        hit = hit.join(r.ranked.set_index("market_id")[["anomaly_score", "anomaly_class"]], how="left")
    return hit[cols(hit, ["market_name", "screen_verdict", "screen_drop_reason", "anomaly_score", "anomaly_class",
                          "close_time", "market_url"])]


if EXPORT_EXCEL:
    target = notebook_folder() / f"falnama_review_{R.id}.xlsx"
    sheets = {"sensor_health": sensor_health(R), "screen_verdicts": R.screen_verdicts, "anomalies": R.ranked,
              "news_lag": pd.DataFrame(R.news_lag).drop(columns=["evidence"], errors="ignore"),
              "card_predictions": CARDS, "rejected_signals": R.rejected_signals,
              "wallet_evidence": R.wallet_evidence, "history": HIST}
    with pd.ExcelWriter(target) as xl:
        for name, frame in sheets.items():
            frame.to_excel(xl, sheet_name=name, index=False)
    print("Saved", target)
else:
    print("Tip: set EXPORT_EXCEL = True in the settings cell to save these tables as a spreadsheet.")